# Noise Sensitivity Analysis

This notebook explores how different noise models affect qMRI parameter estimation.

## Learning Objectives

- Understand Gaussian vs Rician noise
- Analyse SNR impact on fitting accuracy
- Explore optimal acquisition design
- Quantify uncertainty in parameter estimates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qmri.diffusion import adc
from qmri.dro import dwi, relaxometry
from qmri.relaxometry import t1

## 1. Gaussian vs Rician Noise

MRI magnitude images have **Rician noise**, not Gaussian:

$$S_{noisy} = \sqrt{(S + \epsilon_r)^2 + \epsilon_i^2}$$

where $\epsilon_r, \epsilon_i \sim \mathcal{N}(0, \sigma^2)$.

At high SNR, Rician ≈ Gaussian. At low SNR, Rician causes **positive bias**.

In [ ]:
# Compare noise distributions
from qmri.noise import gaussian, rician

true_signal = 100.0
snr = 5  # Low SNR to show difference
n_samples = 10000

gaussian_samples = [
    gaussian.add_noise(true_signal, snr=snr, rng=np.random.default_rng(i))
    for i in range(n_samples)
]
rician_samples = [
    rician.add_noise(true_signal, snr=snr, rng=np.random.default_rng(i))
    for i in range(n_samples)
]

fig, ax = plt.subplots(figsize=(10, 5))

bins = np.linspace(0, 200, 50)
ax.hist(
    gaussian_samples,
    bins=bins,
    alpha=0.5,
    label=f"Gaussian (mean={np.mean(gaussian_samples):.1f})",
    density=True,
)
ax.hist(
    rician_samples,
    bins=bins,
    alpha=0.5,
    label=f"Rician (mean={np.mean(rician_samples):.1f})",
    density=True,
)
ax.axvline(
    true_signal,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"True signal = {true_signal}",
)

ax.set_xlabel("Signal")
ax.set_ylabel("Density")
ax.set_title(f"Noise Distributions at SNR = {snr}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"Gaussian mean: {np.mean(gaussian_samples):.1f} (bias: {np.mean(gaussian_samples) - true_signal:+.1f})"
)
print(
    f"Rician mean: {np.mean(rician_samples):.1f} (bias: {np.mean(rician_samples) - true_signal:+.1f})"
)

## 2. Impact on ADC Fitting

In [ ]:
# Compare noise models for ADC fitting
true_adc = 1.0e-3
snr_levels = [5, 10, 20, 30, 50, 100]
n_simulations = 200

results = {"gaussian": {}, "rician": {}}

for noise_model in ["gaussian", "rician"]:
    for snr in snr_levels:
        adc_values = []
        for seed in range(n_simulations):
            phantom = dwi.generate(
                adc=true_adc, snr=snr, noise_model=noise_model, seed=seed
            )
            result = adc.fit(phantom.signal, phantom.b_values, method="iwlls")
            adc_values.append(result.adc)

        values = np.array(adc_values)
        results[noise_model][snr] = {
            "mean": np.mean(values),
            "bias_pct": 100 * (np.mean(values) - true_adc) / true_adc,
            "cv": 100 * np.std(values) / np.mean(values),
        }

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bias comparison
ax = axes[0]
for noise_model, color in [("gaussian", "blue"), ("rician", "orange")]:
    bias = [results[noise_model][snr]["bias_pct"] for snr in snr_levels]
    ax.plot(
        snr_levels,
        bias,
        "o-",
        color=color,
        linewidth=2,
        markersize=8,
        label=noise_model.capitalize(),
    )

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("SNR")
ax.set_ylabel("Bias (%)")
ax.set_title("ADC Bias: Gaussian vs Rician Noise")
ax.legend()
ax.grid(True, alpha=0.3)

# CV comparison
ax = axes[1]
for noise_model, color in [("gaussian", "blue"), ("rician", "orange")]:
    cv = [results[noise_model][snr]["cv"] for snr in snr_levels]
    ax.plot(
        snr_levels,
        cv,
        "o-",
        color=color,
        linewidth=2,
        markersize=8,
        label=noise_model.capitalize(),
    )

ax.set_xlabel("SNR")
ax.set_ylabel("CV (%)")
ax.set_title("ADC Precision: Gaussian vs Rician Noise")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. SNR Requirements for Different Accuracies

In [ ]:
# Find SNR required for target precision
target_cvs = [5, 10, 15, 20]  # percent

print("SNR Required for Target Precision (IWLLS, Rician noise):")
print("=" * 40)
print(f"{'Target CV (%)':>15} {'Required SNR':>15}")
print("-" * 40)

snr_fine = np.arange(10, 150, 5)
cv_vs_snr = []

for snr in snr_fine:
    adc_values = []
    for seed in range(100):
        phantom = dwi.generate(adc=1e-3, snr=snr, seed=seed)
        result = adc.fit(phantom.signal, phantom.b_values)
        adc_values.append(result.adc)
    cv_vs_snr.append(100 * np.std(adc_values) / np.mean(adc_values))

for target in target_cvs:
    # Find first SNR where CV < target
    for snr, cv in zip(snr_fine, cv_vs_snr):
        if cv <= target:
            print(f"{target:>15} {snr:>15}")
            break

In [ ]:
# Plot CV vs SNR curve
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(snr_fine, cv_vs_snr, "b-", linewidth=2)
ax.axhline(10, color="red", linestyle="--", alpha=0.7, label="10% CV target")
ax.axhline(5, color="green", linestyle="--", alpha=0.7, label="5% CV target")

ax.set_xlabel("SNR")
ax.set_ylabel("Coefficient of Variation (%)")
ax.set_title("ADC Precision vs SNR")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 150])
ax.set_ylim([0, 50])
plt.tight_layout()
plt.show()

## 4. T1 Noise Sensitivity

In [ ]:
# T1 fitting sensitivity
true_t1 = 1.2
snr_levels_t1 = [20, 50, 100, 200]
n_simulations = 100

t1_results = {}
for snr in snr_levels_t1:
    t1_values = []
    for seed in range(n_simulations):
        phantom = relaxometry.generate_t1_ir(
            t1=true_t1,
            inversion_times=[0.1, 0.3, 0.5, 0.8, 1.2, 2.0, 3.0],
            repetition_time=5.0,
            snr=snr,
            seed=seed,
        )
        result = t1.fit_ir(phantom.signal, phantom.time_points, repetition_times=5.0)
        t1_values.append(float(result.t1))

    values = np.array(t1_values)
    t1_results[snr] = {
        "bias_pct": 100 * (np.mean(values) - true_t1) / true_t1,
        "cv": 100 * np.std(values) / np.mean(values),
    }

print("T1 Fitting Sensitivity:")
print("=" * 40)
print(f"{'SNR':>10} {'Bias (%)':>12} {'CV (%)':>12}")
print("-" * 40)
for snr in snr_levels_t1:
    print(
        f"{snr:>10} {t1_results[snr]['bias_pct']:>+12.1f} {t1_results[snr]['cv']:>12.1f}"
    )

## 5. Acquisition Design Optimisation

In [ ]:
# Compare different number of b-values (fixed scan time assumption)
# More b-values = less averages = same total SNR but spread differently

b_value_configs = {
    "3 b-values": (0, 500, 1000),
    "4 b-values": (0, 400, 800, 1200),
    "5 b-values": (0, 300, 600, 900, 1200),
    "6 b-values": (0, 200, 400, 600, 800, 1000),
}

true_adc = 1.0e-3
snr = 30
n_sim = 200

config_results = {}
for name, b_values in b_value_configs.items():
    adc_values = []
    for seed in range(n_sim):
        phantom = dwi.generate(adc=true_adc, b_values=b_values, snr=snr, seed=seed)
        result = adc.fit(phantom.signal, phantom.b_values)
        adc_values.append(result.adc)

    values = np.array(adc_values)
    config_results[name] = {
        "bias_pct": 100 * (np.mean(values) - true_adc) / true_adc,
        "cv": 100 * np.std(values) / np.mean(values),
    }

print("Acquisition Design Comparison (SNR=30):")
print("=" * 50)
print(f"{'Config':<15} {'Bias (%)':>12} {'CV (%)':>12}")
print("-" * 50)
for name in b_value_configs:
    print(
        f"{name:<15} {config_results[name]['bias_pct']:>+12.2f} {config_results[name]['cv']:>12.2f}"
    )

## Summary

Key insights:

1. **Rician noise matters**: Causes positive bias at low SNR
2. **SNR requirements**: ~50 SNR for <10% CV in ADC
3. **More measurements help**: But diminishing returns
4. **DROs enable design**: Optimise protocols before scanning
5. **Always validate**: Test algorithms with known ground truth